# Predicting the next Cambridge 9702 Paper 4 · Colab quick start
This notebook runs the `exam_predictor` package from this repository, one step per cell, so you can see every result.

**How to run:** *Runtime → Change runtime type → T4 GPU*. Put your repository address in `REPO_URL` below, then *Runtime → Run all*.
On a free T4 the first run takes about 1 hour, mostly the AI model writing questions. Everything is cached on Drive, so later runs take a few minutes.

To run everything with a single call instead, use `from exam_predictor.pipeline import run` and then `results = run(settings)`.

## Setup · get the code, install the libraries, connect Drive

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/physics-exam-predictor.git"   # change this to your repository
REPO = "/content/physics-exam-predictor"

!test -d {REPO} || git clone -q {REPO_URL} {REPO}
!pip install -q -r {REPO}/requirements.txt

import sys
sys.path.insert(0, f"{REPO}/src")      # use the code in the repository (no restart needed)

from google.colab import drive
drive.mount("/content/drive")

## Step 1 · Settings
`data_dir` must point at your past-papers folder (subfolders are fine). Results and caches are saved in `work_dir`.

In [ ]:
from exam_predictor.config import Settings

settings = Settings(
    data_dir="/content/drive/MyDrive/physics pastpapers",
    work_dir="/content/drive/MyDrive/physics_exam_project",
    target_exam="October/November 2026",
)
settings

## Step 2 · Load the AI model
About 19 GB is downloaded the first time. Warnings that a fast kernel "falls back to its reference PyTorch implementation" are normal on a T4.

In [ ]:
from exam_predictor.llm import LLM

llm = LLM(settings.llm_model, settings.work_dir)

## Steps 3–6 · Find the papers and build the question table
A normal Paper 4 totals 100 marks. Papers that fail this check are listed below.

In [ ]:
from exam_predictor.dataset import build_question_table, show_question
from exam_predictor.display import show
from exam_predictor.papers import find_papers, papers_per_session

papers = find_papers(settings.data_dir)
show(papers_per_session(papers))
df, quality = build_question_table(papers, settings.syllabus_start, settings.reports_dir)
show(quality[~quality.extraction_ok].head(15))
show_question(df, df.question_id.iat[0], images=False)     # images=True also shows the real PDF pages

## Steps 7–8 · Syllabus outcomes, embeddings, tags and search
**Check:** read the 10 random tags at the end. Count how many you would change; that is the tag accuracy you can report.

In [ ]:
from exam_predictor.embeddings import Embedder, part_vectors
from exam_predictor.search import SearchIndex
from exam_predictor.syllabus import load_outcomes, outcome_info, topic_order
from exam_predictor.tagging import tag_parts

los = load_outcomes(settings.work_dir)
info = outcome_info(los)
embedder = Embedder(settings.embed_model)
part_vecs = part_vectors(df, embedder, settings.work_dir)
df = tag_parts(df, los, part_vecs, embedder, llm if settings.check_tags_with_llm else None, settings.work_dir)
index = SearchIndex(df, part_vecs, embedder)

print("Parts per topic (current syllabus):")
show(df[df.current].topic.value_counts().to_frame("parts"))
show(df[df.current].sample(10, random_state=1)[["question_id", "text", "lo", "lo_text"]])

## Step 9 · Analysis: important topics, repeated tasks and trends
All tables and charts are also saved in `reports/`.

In [ ]:
from exam_predictor.analysis import run_analysis
from exam_predictor.dataset import analysis_pool

pool = analysis_pool(df)
tables = run_analysis(pool, info, topic_order(los), settings.reports_dir, settings.syllabus_start)

## Step 10 · Predict the next paper and backtest the method
The backtest shows how much better the prediction is than random guessing; report these numbers in your project.

In [ ]:
import pandas as pd
from exam_predictor.blueprint import backtest, best_recency, make_blueprint, typical_paper

bt = backtest(pool, info)
show(pd.concat([bt, bt.mean().to_frame("average").T.round(1)]))
recency = best_recency(bt)
blueprint = make_blueprint(pool, info, recency)
print(f"Recency weight {recency}; blueprint shaped like {typical_paper(pool)} ({blueprint.marks.sum()} marks)")
show(blueprint[["question", "topic", "marks", "outcomes", "parts"]], 120)
bt.to_csv(settings.reports_dir / "backtest.csv")
blueprint.to_csv(settings.reports_dir / "blueprint.csv", index=False)

## Step 11 · Write the predicted paper
Each attempt prints one line with its problems; problems found by the review start with `examiner:`. About 1–2 minutes per attempt on a T4.

In [ ]:
from exam_predictor.generation.writer import summary_table, write_paper

generated = write_paper(blueprint, pool, llm, embedder, part_vecs,
                        settings.max_tries, settings.copy_limit, settings.review)
show(summary_table(generated))

## Step 12 · Show and save the paper and mark scheme
Saved as Word and Markdown files in `reports/`. Anything the checks could not fix is printed as **CHECK**. Solve the paper yourself before trusting it.

In [ ]:
from exam_predictor.export import save_paper, show_paper

save_paper(generated, settings.reports_dir, settings.target_exam)
show_paper(generated, settings.reports_dir, settings.target_exam)

## Step 13 · Ask the archive (RAG question answering)
Change the question and re-run this cell as often as you like.

In [ ]:
from exam_predictor.archive import ask_archive

ask_archive("How do mark schemes award marks for questions on Wien's displacement law?", index, llm,
            years=(settings.syllabus_start, 2100))
show(index.similar(pool.question_id.iat[0]))